用于引导用户构建会学习的 Mario 的 Pre-MVP 教程。创建此 notebook 的准则（可随意添加/编辑）：
1. 提供充分说明（必要时链接到 AI 速查表）
2. 只要求实现核心逻辑


In [1]:
from IPython.display import HTML
from IPython import display as ipythondisplay
import glob
import io
import base64
import os
import shutil
import sys

# pyvirtualdisplay 需要系统里安装 Xvfb；Windows 本地环境通常没有也不需要它。
# 因此只在非 Windows 且能找到 Xvfb 时启动虚拟显示，避免 WinError 2。
display = None
if sys.platform != "win32" and shutil.which("Xvfb"):
    from pyvirtualdisplay import Display
    display = Display(visible=0, size=(1400, 900))
    display.start()
else:
    print("未启动虚拟显示：当前环境不需要或未安装 Xvfb。")

"""
用于开启 gymnasium 环境视频录制并显示视频的工具函数。
要开启视频，只需执行 "env = wrap_env(env)"。
"""

def show_video():
	mp4list = glob.glob('../video/*.mp4')
	if len(mp4list) > 0:
		mp4 = mp4list[0]
		video = io.open(mp4, 'r+b').read()
		encoded = base64.b64encode(video)
		ipythondisplay.display(HTML(data='''<video alt="test" autoplay 
					loop controls style="height: 400px;">
					<source src="data:video/mp4;base64,{0}" type="video/mp4" />
					</video>'''.format(encoded.decode('ascii'))))
	else: 
		print("找不到视频")
    

未启动虚拟显示：当前环境不需要或未安装 Xvfb。


In [2]:
import torch.nn as nn
import numpy as np

class ConvNet(nn.Module):
    '''迷你 CNN 结构
    输入 -> (conv2d + relu) x 3 -> 展平 -> (dense + relu) x 2 -> 输出
    '''
    def __init__(self, input_dim, output_dim):
        super(ConvNet, self).__init__()
        c, h, w = input_dim
        self.conv_1 = nn.Conv2d(in_channels=c, out_channels=32, kernel_size=8, stride=4)
        self.conv_2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=4, stride=2)
        self.conv_3 = nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, stride=1)
        self.relu = nn.ReLU()
        self.flatten = nn.Flatten()
        self.dense = nn.Linear(3136, 512)
        self.output = nn.Linear(512, output_dim)

    def forward(self, input):
        # input: B x C x H x W
        x = input
        x = self.conv_1(x)
        x = self.relu(x)
        x = self.conv_2(x)
        x = self.relu(x)
        x = self.conv_3(x)
        x = self.relu(x)

        x = self.flatten(x)
        x = self.dense(x)
        x = self.relu(x)
        x = self.output(x)

        return x


# 欢迎来到 Mad Mario！

我们整理了这个项目，带你了解强化学习的基础知识。在项目过程中，你将实现一个聪明的 Mario，让它学会自己通关。刚开始时，你不需要了解任何强化学习（RL）知识。如果你想提前浏览，这里有一份 [RL 基础速查表](https://colab.research.google.com/drive/1eN33dPVtdPViiS1njTW_-r-IYCDTFU7N?usp=sharing)，我们会在整个项目中反复引用。完成本教程后，你将扎实理解 RL 基础，并亲自实现一个经典 RL 算法：Q-learning。


建议你熟悉 Python，并具备高中或同等水平的数学/统计学基础——不过即使记不太清也不用担心。只要在任何让你困惑的地方留下评论，我们会更详细地解释对应部分。

## 开始吧！

首先，让我们看看我们要构建什么：就像我们第一次玩游戏时一样，Mario 进入游戏时对游戏一无所知。它会随机采取动作，只是为了更好地理解游戏。每一次失败经验都会加入 Mario 的记忆；随着失败不断积累，Mario 开始识别在特定场景下应该采取的更好动作。最终，Mario 会学到一个不错的策略并完成关卡。

让我们把这个故事写成伪代码。

```
总共进行 N 个回合：
  每个回合总共进行 M 个步骤：
    Mario 执行一个动作
    游戏给出反馈
    Mario 记住这个动作和反馈
    在积累了一些经验之后：
      Mario 从经验中学习
```



用 RL 术语来说：智能体（Mario）通过选择动作与环境（游戏）交互，环境则返回奖励和下一个状态。基于收集到的（状态、动作、奖励）信息，智能体会通过优化动作策略来最大化未来回报。

这些术语一开始可能听起来有点吓人，但很快你就会理解它们。在开始写代码之前，先复习一下这份[速查表](https://colab.research.google.com/drive/1eN33dPVtdPViiS1njTW_-r-IYCDTFU7N?usp=sharing)会很有帮助。我们将从“环境”这个概念开始本教程。



# 环境
[环境（Environment）](https://colab.research.google.com/drive/1eN33dPVtdPViiS1njTW_-r-IYCDTFU7N#scrollTo=OMfuO883blEq) 是强化学习中的关键概念。它是 Mario 与之交互并从中学习的世界。环境由[状态（state）](https://colab.research.google.com/drive/1eN33dPVtdPViiS1njTW_-r-IYCDTFU7N#scrollTo=36WmEZ-8bn9M)刻画。在 Mario 中，这就是由管道、蘑菇和其他组件组成的游戏画面。当 Mario 执行动作时，环境会返回一个[奖励（reward）](https://colab.research.google.com/drive/1eN33dPVtdPViiS1njTW_-r-IYCDTFU7N#scrollTo=rm0EqRQqbo09)和[下一个状态（next state）](https://colab.research.google.com/drive/1eN33dPVtdPViiS1njTW_-r-IYCDTFU7N#scrollTo=_SnLbEzua1pv)。  


让我们尝试运行 Mario 环境。






In [3]:
import gymnasium as gym
from gymnasium.wrappers import RecordVideo
from gymnasium.wrappers import FrameStackObservation as FrameStack, GrayscaleObservation as GrayScaleObservation
from gymnasium.spaces import Box
import gym_super_mario_bros
from nes_py.wrappers import JoypadSpace
import gym_super_mario_bros
from skimage import transform
import torch

In [4]:
env = gym_super_mario_bros.make('SuperMarioBros-1-1-v0', render_mode='rgb_array')
"""
在我们的环境中，Mario 可以执行两个动作：向右走和向右跳。
如需查看所有可能动作的完整列表，请参见：
https://github.com/Kautenja/gym-super-mario-bros/blob/master/gym_super_mario_bros/actions.py
"""
env = JoypadSpace(
    env,
    [['right'],
    ['right', 'A']]
)
env = RecordVideo(env, video_folder='../video', episode_trigger=lambda episode_id: True)
# 重启环境
env.reset()
for _ in range(1000):
	# 渲染视频输出
	env.render()

	# 选择随机动作
	action = env.action_space.sample()

	# 在环境中执行动作，环境会通过 env.step() 函数提供反馈
	next_state, reward, terminated, truncated, info = env.step(action=action)
	done = terminated or truncated
	if done:
		break

# 关闭环境
env.close()

show_video()

c:\Users\swix\Code\PythonProjects\MadMario\.venv\Lib\site-packages\gymnasium\wrappers\rendering.py:292: UserWarning: WARN: Overwriting existing videos at c:\Users\swix\Code\PythonProjects\MadMario\video folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


## 包装器

很多时候，在把环境数据输入给智能体之前，我们希望先对环境做一些预处理。这就引出了包装器（wrapper）的概念。

一个常见包装器会把 RGB 图像转换为灰度图。这可以在不损失太多信息的情况下减小状态表示的大小。对于智能体来说，它生活在 RGB 世界还是灰度世界，并不会改变它的行为！

**包装前**

![wrapper_before](../img/wrapper_before.png)

**包装后**

![wrapper_after](../img/wrapper_after.jpg)

我们可以用这种方式把包装器应用到环境上：
```
env = wrapper(env, **args)
```


### 说明

我们希望给定的 `env` 应用 3 个内置包装器：`GrayScaleObservation`、`ResizeObservation` 和 `FrameStack`。

https://gymnasium.farama.org/api/wrappers/


`FrameStack` 是一个包装器，它允许我们把环境中的连续帧压缩成一个单独的观测点，并输入到学习模型中。这样，我们可以根据 Mario 在前几帧中的运动方向，区分它是在落地还是跳跃。

让我们使用以下参数：
`GrayScaleObservation`: keep_dim=False  
`ResizeObservation`: shape=84  
`FrameStack`: stack_size=4




In [5]:


class ResizeObservation(gym.ObservationWrapper):
    """将图像观测下采样为正方形图像。"""
    def __init__(self, env, shape):
        super().__init__(env)
        if isinstance(shape, int):
            self.shape = (shape, shape)
        else:
            self.shape = tuple(shape)

        obs_shape = self.shape + self.observation_space.shape[2:]
        self.observation_space = Box(low=0, high=255, shape=obs_shape, dtype=np.uint8)

    def observation(self, observation):
        resize_obs = transform.resize(observation, self.shape)
        resize_obs *= 255
        return resize_obs.astype(np.uint8)

# 原始环境对象
env = gym_super_mario_bros.make('SuperMarioBros-1-1-v0')
env = JoypadSpace(
    env,
    [['right'],
    ['right', 'A']]
)

# TODO 使用 GrayScaleObservation 包装给定的 env
env = GrayScaleObservation(env, keep_dim=False)
# TODO 使用 ResizeObservation 包装给定的 env
env = ResizeObservation(env, shape=84)
# TODO 使用 FrameStack 包装给定的 env
env = FrameStack(env, stack_size=4)

## 自定义包装器

我们也希望你体验一下自己实现环境包装器，而不是直接调用现成的软件包。

这里有一个想法：为了加快训练速度，我们可以跳过一些帧，只显示每第 n 帧。虽然跳过了一些帧，但累加这些被跳过帧中的所有奖励非常重要。把所有中间奖励相加，并在第 n 帧返回。


### 说明

我们的自定义包装器 `SkipFrame` 继承自 `gymnasium.Wrapper`，并且需要实现 `step()` 函数。

在 for 循环中每个被跳过的帧里，将 `reward` 累加到 `total_reward`，并且如果任意一步返回 `terminated=True` 或 `truncated=True`，就中断循环。

In [6]:
class SkipFrame(gym.Wrapper):
    def __init__(self, env, skip=4):
        """只返回每第 `skip` 帧"""
        super().__init__(env)
        self._skip = skip

    def step(self, action):
        """重复执行动作，并累加奖励"""
        total_reward = 0.0
        terminated = False
        truncated = False
        for i in range(self._skip):
            # TODO 累加奖励并重复相同动作
            obs, reward, terminated, truncated, info = self.env.step(action)
            total_reward += reward
            if terminated or truncated:
                break

        return obs, total_reward, terminated, truncated, info

env = SkipFrame(env)

**最终包装后的状态**

![picture](../img/tensor.png)

将上述包装器应用到环境之后，最终包装后的状态由 4 个连续的灰度帧堆叠而成，如上方左侧图片所示。每当 Mario 执行一个动作时，环境都会返回一个具有这种结构的状态。该结构由一个大小为 (4 * 84 * 84) 的三维数组表示。


# 智能体

现在让我们转向强化学习中的另一个核心概念：[智能体（agent）](https://colab.research.google.com/drive/1eN33dPVtdPViiS1njTW_-r-IYCDTFU7N#scrollTo=OMfuO883blEq)。智能体通过按照自己的[动作策略（action policy）](https://colab.research.google.com/drive/1eN33dPVtdPViiS1njTW_-r-IYCDTFU7N#scrollTo=_SnLbEzua1pv)执行[动作（actions）](https://colab.research.google.com/drive/1eN33dPVtdPViiS1njTW_-r-IYCDTFU7N#scrollTo=chyu7AVObwWP)，与环境进行交互。让我们回顾一下智能体如何与环境交互的伪代码：

```
总共进行 N 个回合：
每个回合总共进行 M 个步骤：
  Mario 执行一个动作
  游戏给出反馈
  Mario 记住这个动作和反馈
  在积累了一些经验之后：
    Mario 从经验中学习
```





我们创建一个 `Mario` 类来表示游戏中的智能体。`Mario` 应该能够：

- 选择要执行的动作。`Mario` 会基于当前环境[状态（state）](https://colab.research.google.com/drive/1eN33dPVtdPViiS1njTW_-r-IYCDTFU7N#scrollTo=36WmEZ-8bn9M)，按照自己的[最优动作策略（optimal action policy）](https://colab.research.google.com/drive/1eN33dPVtdPViiS1njTW_-r-IYCDTFU7N#scrollTo=SZ313skqbSjQ)行动。

- 记住经验。经验由当前环境状态、当前智能体动作、环境给出的奖励以及下一个环境状态组成。`Mario` 之后会使用所有这些经验来更新自己的[动作策略（action policy）](https://colab.research.google.com/drive/1eN33dPVtdPViiS1njTW_-r-IYCDTFU7N#scrollTo=_SnLbEzua1pv)。

- 随着时间推移改进动作策略。`Mario` 使用 [Q-learning](https://colab.research.google.com/drive/1eN33dPVtdPViiS1njTW_-r-IYCDTFU7N#scrollTo=bBny3BgNbcmh) 更新自己的动作策略。

在接下来的章节中，我们会交替使用 Mario 和智能体这两个说法。

In [7]:
class Mario:
    def __init__(self, state_dim, action_dim):
        pass

    def act(self, state):
        """给定一个状态，选择一个 epsilon-greedy 动作
        """
        pass

    def remember(self, experience):
        """将观测添加到记忆中
        """
        pass

    def learn(self):
        """使用一批经验更新在线动作价值（Q）函数
        """
        pass


## 初始化
在实现上述任何函数之前，让我们先定义一些关键参数。

### 说明

在 `__init__()` 中初始化这些关键参数。

> `exploration_rate: float = 1.0`

随机探索概率。在[某个概率 $\epsilon$](https://colab.research.google.com/drive/1eN33dPVtdPViiS1njTW_-r-IYCDTFU7N#scrollTo=_SnLbEzua1pv) 下，智能体不会遵循[最优动作策略](https://colab.research.google.com/drive/1eN33dPVtdPViiS1njTW_-r-IYCDTFU7N#scrollTo=SZ313skqbSjQ)，而是选择一个随机动作来探索环境。在学习早期，较高的探索率很重要，它能确保充分探索并避免陷入局部最优。随着智能体策略的改进，探索率应逐渐降低。


> `exploration_rate_decay: float = 0.99999975`

`exploration_rate` 的衰减率。智能体在早期会充分探索空间，但会逐渐降低探索率以保持动作质量。在后期，智能体已经学到了相当不错的策略，所以我们希望它更频繁地遵循自己的策略。每次智能体行动后，都用 `exploration_rate_decay` 这个因子来降低 `exploration_rate`。

> `exploration_rate_min: float = 0.1`

Mario 的 `exploration_rate` 能衰减到的最小值。注意，这个值也可以是 `0`，此时 Mario 的行动会完全确定；也可以是一个非常小的数。

> `discount_factor: float = 0.9`

未来奖励折扣因子。这就是[回报（return）](https://colab.research.google.com/drive/1eN33dPVtdPViiS1njTW_-r-IYCDTFU7N#scrollTo=6lmIKuxsb6qu)定义中的 $\gamma$。它用于让智能体相比未来奖励，更重视短期奖励。


> `batch_size: int = 32`

每次更新所使用的经验数量。


> `state_dim`

状态空间维度。在 Mario 中，它是 4 个连续环境快照堆叠在一起，其中每个快照都是一张 84*84 的灰度图。它由环境传入：`self.state_dim = (4, 84, 84)`。

> `action_dim`

动作空间维度。在 Mario 中，它是所有可能动作的总数。它同样由环境传入。

> `memory`

`memory` 是一个队列结构，里面装着 Mario 过去的经验。每条经验由 (state, next_state, action, reward, done) 组成。随着 Mario 收集到更多经验，旧经验会被弹出，以便为最近的经验腾出空间。我们用 `maxlen=100000` 初始化这个记忆队列。

In [8]:
from collections import deque

class Mario(object):
    def __init__(self, state_dim, action_dim):
        # 状态空间维度
        self.state_dim = state_dim
        # 动作空间维度
        self.action_dim = action_dim
        # 回放缓冲区
        self.memory = deque(maxlen=100000)
        # 当前步数，每次智能体行动时更新
        self.step = 0

        # TODO：请按照上面的描述初始化其他变量
        self.exploration_rate = 1.0
        self.exploration_rate_decay = 0.99999975
        self.exploration_rate_min = 0.1
        self.discount_factor = 0.9
        self.batch_size = 32
    


## 预测 $Q^*$

[最优动作价值函数（Optimal value action function）](https://colab.research.google.com/drive/1eN33dPVtdPViiS1njTW_-r-IYCDTFU7N#scrollTo=snRMrCIccEx8) $Q^*(s, a)$ 是本项目中最重要的函数。`Mario` 用它来选择[最优动作](https://colab.research.google.com/drive/1eN33dPVtdPViiS1njTW_-r-IYCDTFU7N#scrollTo=SZ313skqbSjQ)

$$
a^*(s) = argmax_{a}Q^*(s, a)
$$

并[更新自己的动作策略](https://colab.research.google.com/drive/1eN33dPVtdPViiS1njTW_-r-IYCDTFU7N#scrollTo=bBny3BgNbcmh)

$$
Q^*(s, a) \leftarrow Q^*(s, a)+\alpha (r + \gamma \max_{a'} Q^*(s', a') -Q^*(s, a))
$$

在本节中，让我们实现 `agent.predict()` 来计算 $Q^*(s, a)$。


### $Q^*_{online}$ 和 $Q^*_{target}$

让我们回顾一下 $Q^*(s, a)$ 函数的输入。

$s$ 是从环境中观测到的状态。经过我们的包装器处理后，$s$ 是一组灰度图像的堆叠。$a$ 是一个表示所采取动作的整数。为了处理图像/视频信号，我们通常会使用[*卷积神经网络*](https://pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html)。为了节省你的时间，我们已经为你创建了一个简单的 [`ConvNet`](https://colab.research.google.com/drive/1kptUkdESbxBC-yOfSYngynjV5Hge_-t-#scrollTo=M27B_D2WEQ22)。

我们并不是把动作 $a$ 和状态 $s$ 一起传入 $Q^*$ 函数，而是只传入状态。`ConvNet` 会返回一个实数列表，表示*所有动作*对应的 $Q^*$。之后我们可以选择任意特定动作 $a$ 的 $Q^*$。

<!-- 现在让我们更仔细地看看 Q-learning。

$$
Q^*(s, a) \leftarrow Q^*(s, a)+\alpha (r + \gamma \max_{a'} Q^*(s', a') -Q^*(s, a))
$$

$(r + \gamma \max_{a'} Q^*(s', a'))$ 是 *TD 目标*（速查表），而 $Q^*(s, a)$ 是 *TD 估计*（速查表）。$s$ 和 $a$ 是当前状态和动作，$s'$ 和 $a'$ 是下一个状态和下一个动作。 -->

在本节中，我们定义两个函数：$Q_{online}$ 和 $Q_{target}$。二者都表示最优动作价值函数 $Q^*$。直观地说，我们使用 $Q_{online}$ 来做动作决策，使用 $Q_{target}$ 来改进 $Q_{online}$。我们会在[后续章节](https://colab.research.google.com/drive/1kptUkdESbxBC-yOfSYngynjV5Hge_-t-#scrollTo=BOALqrSC5VIf)进一步详细解释。



### 说明

使用我们提供的 [`ConvNet`](https://colab.research.google.com/drive/1kptUkdESbxBC-yOfSYngynjV5Hge_-t-#scrollTo=M27B_D2WEQ22) 分别定义 `self.online_q` 和 `self.target_q`。对两个 $Q^*$ 函数，都使用 `input_dim=self.state_dim` 和 `output_dim=self.action_dim` 初始化 `ConvNet`。


In [9]:
class Mario(Mario):
	def __init__(self, state_dim, action_dim):
		super().__init__(state_dim, action_dim)
		# TODO：定义在线动作价值函数
		self.online_q = ConvNet(input_dim=self.state_dim, output_dim=self.action_dim)
		# TODO：定义目标动作价值函数
		self.target_q = ConvNet(input_dim=self.state_dim, output_dim=self.action_dim)

### 调用 $Q^*$

### 说明

`self.online_q` 和 `self.target_q` 都是最优动作价值函数 $Q^*$，它们接收单个输入 $s$。

实现 `Mario.predict()` 来计算输入 $s$ 的 $Q^*$。这里，$s$ 是一批状态，即：
```
shape(state) = batch_size, 4, 84, 84
```

返回整批状态中所有可能动作对应的 $Q^*$。


In [10]:
class Mario(Mario):
	def predict(self, state, model):
		"""
		给定一个状态，使用指定模型（online 或 target）预测所有可能动作的 Q 值
		输入：
		state
		维度为 (batch_size * state_dim)
		model
		可以是 'online' 或 'target'
		输出：
		pred_q_values (torch.tensor)
			维度为 (batch_size * action_dim)，表示给定状态下所有可能动作的预测 Q 值
		"""
		# LazyFrame -> np array -> torch tensor
		state_float = torch.FloatTensor(np.array(state))
		# 归一化
		state_float = state_float / 255.
		
		if model == 'online':
			# TODO 使用 self.online_q 返回预测 Q 值
			pred_q_values = self.online_q(state_float)
		elif model == 'target':
			# TODO 使用 self.target_q 返回预测 Q 值
			pred_q_values = self.target_q(state_float)

		return pred_q_values

## 行动

现在让我们看看 Mario 应该如何在环境中 `act()`。

给定一个状态，Mario 大多数时候会[选择 Q 值最高的动作](https://colab.research.google.com/drive/1eN33dPVtdPViiS1njTW_-r-IYCDTFU7N#scrollTo=SZ313skqbSjQ)。同时，Mario 有一个 *epsilon* 概率会改为随机行动，这鼓励它探索环境。

### 说明

本节会使用 `torch.tensor` 和 `numpy.array`。请通过[一些基础语法示例](https://colab.research.google.com/drive/1D8k6i-TIKfqEHVkzKwYMjJvZRAKe9IuH?usp=sharing)熟悉它们。

现在我们将实现 `Mario.act()`。回忆一下，我们上面定义了 $Q_{online}$，这里会使用它根据给定*状态*计算所有动作的 Q 值。然后我们需要选择产生最大 Q 值的动作。我们已经搭好了 epsilon-greedy 策略的逻辑，并把确定最优动作和随机动作的部分留给你完成。

在实现 `Mario.act()` 之前，让我们先熟悉一下 *torch.tensor* 的基本操作；它是 `Mario.predict()` 返回的数据类型。
     

In [11]:
class Mario(Mario):
    def act(self, state):
        """给定一个状态，选择一个 epsilon-greedy 动作，并更新 step 的值
        输入：
          state(np.array) 
            当前状态的单个观测，维度为 (state_dim)
        输出：
          action
            表示智能体将执行哪个动作的整数
        """
        if np.random.rand() < self.exploration_rate:
          # TODO：从所有可能动作 (self.action_dim) 中选择一个随机动作
          action = np.random.randint(self.action_dim)
        else:
          state = np.expand_dims(state, 0)
          # TODO：基于 self.online_q 选择最佳动作
          action_values = self.predict(state, model='online')
          action = torch.argmax(action_values, axis=1).item()
          
        # 降低 exploration_rate
        self.exploration_rate *= self.exploration_rate_decay
        self.exploration_rate = max(self.exploration_rate_min, self.exploration_rate)
        # 递增 step
        self.step += 1
        return action

## 记忆

为了改进策略，Mario 需要收集并保存过去的经验。每当智能体执行一个动作时，它都会收集一条经验，其中包括当前状态、它执行的动作、执行动作后的下一个状态、获得的奖励，以及游戏是否结束。

我们使用上面定义的 `self.memory` 来存储过去的经验，经验由 (state, next_state, action, reward, done) 组成。

### 说明

实现 `Mario.remember()`，将经验保存到 Mario 的记忆中。

In [12]:
class Mario(Mario):
    def remember(self, experience):
        """将经验添加到 self.memory
        输入：
          experience =  (state, next_state, action, reward, done) 元组
        输出：
            None
        """
        # TODO 将经验添加到记忆中
        self.memory.append(experience)

## 学习


整个学习过程基于 [Q-learning 算法](https://colab.research.google.com/drive/1eN33dPVtdPViiS1njTW_-r-IYCDTFU7N#scrollTo=bBny3BgNbcmh)。这里所说的学习，是指更新我们的 $Q^*$ 函数，使它能更好地预测当前状态-动作对的最优价值。本节会同时使用 $Q^*_{online}$ 和 $Q^*_{target}$。


需要执行的一些关键步骤：
- **经验采样：**  
我们会从记忆中采样经验，作为*训练数据*来更新 $Q^*_{online}$。


- **评估 TD 估计：**
使用当前状态和动作，计算采样经验的 [TD 估计](https://colab.research.google.com/drive/1eN33dPVtdPViiS1njTW_-r-IYCDTFU7N#scrollTo=2abP5k2kcRnn)。在这一步中，我们使用 $Q^*_{online}$ 直接预测 $Q^*(s, a)$。


- **评估 TD 目标：**
使用下一个状态和奖励，计算采样经验的 [TD 目标](https://colab.research.google.com/drive/1eN33dPVtdPViiS1njTW_-r-IYCDTFU7N#scrollTo=Q072-fLecSkb)。我们同时使用 $Q^*_{online}$ 和 $Q^*_{target}$ 来计算 $r + \gamma \max_{a'} Q^*_{target}(s', a')$，其中 $\max_{a'}$ 部分由 $Q^*_{online}$ 决定。


- **TD 估计与 TD 目标之间的损失：**
计算 TD 估计和 TD 目标之间的均方误差损失。


- **更新 $Q^*_{online}$：**
使用上面计算出的损失执行一次优化步骤，以更新 $Q^*_{online}$。


将上述内容总结为 `Mario.learn()` 的伪代码：

```
如果已经收集到足够多的经验：
  采样一批经验
  使用 Q_online 计算预测 Q 值
  使用 Q_target 和 reward 计算目标 Q 值
  计算预测 Q 值与目标 Q 值之间的损失
  基于损失更新 Q_online
```

### 经验采样
Mario 通过从记忆中抽取过去的经验来学习。记忆是一个队列数据结构，它以如下格式存储每一条独立经验：

> state, next_state, action, reward, done

Mario 记忆中的一些经验示例：


- state: ![pic](https://drive.google.com/uc?id=1D34QpsmJSwHrdzszROt405ZwNY9LkTej)  next_state: ![pic](https://drive.google.com/uc?id=13j2TzRd1SGmFru9KJImZsY9DMCdqcr_J) action: jump reward: 0.0 done: False


- state: ![pic](https://drive.google.com/uc?id=1ByUKXf967Z6C9FBVtsn_QRnJTr9w-18v) next_state: ![pic](https://drive.google.com/uc?id=1hmGGVO1cS7N7kdcM99-K3Y2sxrAFd0Oh) action: right reward: -10.0 done: True


- state: ![pic](https://drive.google.com/uc?id=10MHERSI6lap79VcZfHtIzCS9qT45ksk-) next_state: ![pic](https://drive.google.com/uc?id=1VFNOwQHGAf9pH_56_w0uRO4WUJTIXG90) action: right reward: -10.0 done: True


- state: ![pic](https://drive.google.com/uc?id=1T6CAIMzNxeZlBTUdz3sB8t_GhDFbNdUO) next_state: ![pic](https://drive.google.com/uc?id=1aZlA0EnspQdcSQcVxuVmaqPW_7jT3lfW) action: jump_right reward: 0.0 done: False


- state: ![pic](https://drive.google.com/uc?id=1bPRnGRx2c1HJ_0y_EEOFL5GOG8sUBdIo) next_state: ![pic](https://drive.google.com/uc?id=1qtR4qCURBq57UCrmObM6A5-CH26NYaHv) action: right reward: 10.0 done: False

State/next_state：
时间步 *t*/*t+1* 的观测。它们的类型都是 `LazyFrame`。

Action：
Mario 在状态转移期间采取的动作。

Reward：
状态转移期间环境给出的奖励。

Done：
表示 next_state 是否为终止状态（游戏结束）的布尔值。终止状态的 Q 值已知为 0。


## 说明

从 `self.memory` 中采样一批大小为 `self.batch_size` 的经验。

按 (state, next_state, action, reward, done) 的顺序返回一个 numpy 数组元组。每个 numpy 数组的第一维都应该等于 `self.batch_size`。

要将 `LazyFrame` 转换为 numpy 数组，请执行：

```
state_np_array = np.array(state_lazy_frame)
```

 

In [13]:
import random

class Mario(Mario):
	def sample_batch(self):
		"""
		输入：
			self.memory (FIFO 队列)
			队列中的每一项都包含以下五个元素
			state: 维度为 (state_dim) 的 LazyFrame
			next_state: 维度为 (state_dim) 的 LazyFrame
			action: 整数，表示采取的动作
			reward: 浮点数，从 state 到 next_state 并执行 action 得到的奖励
			done: 布尔值，表示 state 是否为终止状态
			self.batch_size (int)
			要返回的批次大小

		输出：
			state, next_state, action, reward, done (tuple)
			numpy 数组组成的元组：state, next_state, action, reward, done
			state: 维度为 (batch_size x state_dim) 的 numpy 数组
			next_state: 维度为 (batch_size x state_dim) 的 numpy 数组
			action: 维度为 (batch_size) 的 numpy 数组
			reward: 维度为 (batch_size) 的 numpy 数组
			done: 维度为 (batch_size) 的 numpy 数组
		"""
		# TODO 将所有内容转换为 numpy 数组
		batch = random.sample(self.memory, self.batch_size)
		state, next_state, action, reward, done = map(np.array, zip(*batch))
		return state, next_state, action, reward, done

### TD 估计

[*TD 估计*](https://colab.research.google.com/drive/1eN33dPVtdPViiS1njTW_-r-IYCDTFU7N#scrollTo=2abP5k2kcRnn) 是基于*当前状态-动作对 $s, a$* 对 $Q^*(s, a)$ 的估计。

<!-- 它代表我们目前拥有的最佳估计，目标是持续使用 TD 目标（链接到 Q learning 方程）来更新它。我们会使用 $Q_{online}$ 来计算它。 -->

回忆一下我们上面定义的 `Mario.predict()`：
```
q_values = self.predict(state, model='online')
```


## 说明

使用上面定义的 `Mario.predict()`，通过 `online` 模型计算给定 `state` 和 `action` 的 *TD Estimate*。以 `torch.tensor` 格式返回结果。

注意，`Mario.predict()` 返回的是所有动作的 $Q^*$。要定位特定动作对应的 $Q^*$ 值，请使用 [tensor 索引](https://colab.research.google.com/drive/1D8k6i-TIKfqEHVkzKwYMjJvZRAKe9IuH?usp=sharing)。


In [14]:
class Mario(Mario):  
	def calculate_prediction_q(self, state, action):
		"""
		输入：
			state (np.array)
			维度为 (batch_size x state_dim)，每一项都是当前状态的一个观测
			action (np.array)
			维度为 (batch_size)，每一项都是表示当前状态下所采取动作的整数

		输出：
			pred_q (torch.tensor)
			维度为 (batch_size)，每一项都是当前状态-动作对的预测 Q 值
		"""
		curr_state_q = self.predict(state, model='online')
		# TODO 根据输入动作选择对应的 Q 值
		curr_state_q = curr_state_q[np.arange(0, self.batch_size), action]

		return curr_state_q

### TD 目标

[*TD 目标*](https://colab.research.google.com/drive/1eN33dPVtdPViiS1njTW_-r-IYCDTFU7N#scrollTo=Q072-fLecSkb) 是*基于下一个状态-动作对 $s', a'$ 以及奖励 $r$* 对 $Q^*(s, a)$ 的估计。

*TD 目标*的形式为：

$$
r + \gamma \max_{a'} Q^*(s', a')
$$

$r$ 是当前奖励，$s'$ 是下一个状态，$a'$ 是下一个动作。

### 注意事项


**获取最佳下一个动作**

因为我们不知道下一个动作 $a'$ 会是什么，所以我们使用下一个状态 $s'$ 和 $Q_{online}$ 来估计它。具体来说：

$$
a' = argmax_a Q_{online}(s', a)
$$

也就是说，我们把 $Q_{online}$ 应用到 next_state $s'$ 上，选择会产生最大 Q 值的动作，然后用该动作去索引 $Q_{target}$，从而计算 *TD 目标*。这就是为什么如果你比较 `calculate_target_q` 和 `calculate_pred_q` 的函数签名，会发现 `calculate_prediction_q` 中有 `action` 和 `state` 作为输入参数，而 `calculate_target_q` 中只有 `reward` 和 `next_state`。


**终止状态**

另一个小注意事项是终止状态，它由变量 `done` 记录；当 Mario 死亡或游戏结束时，`done` 为 1。

因此，我们需要确保当“没有未来”时，也就是游戏到达终止状态时，不再继续累加未来奖励。由于 `done` 是布尔值，我们可以将 `1.0 - done` 与未来奖励相乘。这样，终止状态之后的未来奖励就不会被计入 TD 目标。

因此，完整的 *TD 目标*形式为：

$$
r + (1.0 - done) \gamma \max_{a'} Q^*_{target}(s', a')
$$
其中 $a'$ 由下式决定：

$$
a' = argmax_a Q_{online}(s', a)
$$

现在让我们计算 *TD 目标*。

## 说明

对于一批由 next_states $s'$ 和 rewards $r$ 组成的经验，计算 *TD 目标*。注意，$a'$ 并没有被显式给出，因此我们需要先使用 $Q_{online}$ 和下一个状态 $s'$ 得到它。

以 `torch.tensor` 格式返回结果。

In [15]:
class Mario(Mario):  
	def calculate_target_q(self, next_state, reward):
		"""
		输入：
			next_state (np.array)
			维度为 (batch_size x state_dim)，每一项都是下一个状态的一个观测
			reward (np.array)
			维度为 (batch_size)，每一项都是表示从 (state -> next state) 转移中获得奖励的浮点数
		输出：
			target_q (torch.tensor)
			维度为 (batch_size)，每一项都是当前状态-动作对的目标 Q 值，
			它基于获得的奖励和对下一个状态的估计 Q 值计算得到
		"""
		next_state_q = self.predict(next_state, 'target')

		online_q = self.predict(next_state, 'online')
		# TODO 基于 online Q 函数选择下一个状态的最佳动作
		action_idx = torch.argmax(online_q, axis=1)

		# TODO 基于 action_idx 和 reward 计算目标 Q 值
		target_q = torch.tensor(reward) + (1. - done) * next_state_q[np.arange(0, self.batch_size), action_idx] * self.gamma

		return target_q

### 损失

现在让我们计算 TD 目标与 TD 估计之间的损失。损失会驱动优化过程，并更新 $Q^*_{online}$，使其未来能更好地预测 $Q^*$。我们将按如下形式计算均方误差损失：

$MSE = \frac{1}{n}\sum_{i=0}^n( y_i - \hat{y}_i)^2$

PyTorch 已经实现了这个损失：
```
loss = nn.functional.mse_loss(pred_q, target_q)
```

## 说明

给定一批经验对应的 *TD 估计*（`pred_q`）和 *TD 目标*（`target_q`），返回均方误差。 


In [16]:
import torch.nn as nn

class Mario(Mario):
	def calculate_loss(self, pred_q, target_q):
		"""
		输入：
		pred_q (torch.tensor)
			维度为 (batch_size)，每一项都是一个观测
		target_q (torch.tensor)
			维度为 (batch_size)，每一项都是表示从 (state -> next state) 转移中获得奖励的浮点数

		输出：
		loss (torch.tensor)
			一个单值，表示 pred_q 和 target_q 的 MSE 损失
		"""
		# TODO 计算均方误差损失
		loss = nn.functional.mse_loss(pred_q, target_q)
		return loss

### 更新 $Q^*_{online}$

作为完成 `Mario.learn()` 的最后一步，我们使用 Adam 优化器基于上面计算出的 `loss` 进行优化。这会更新 $Q^*_{online}$ 函数中的参数，使 TD 估计更接近 TD 目标。

到目前为止你已经写了很多代码。这一节我们已经帮你处理好了。

In [17]:
import torch

class Mario(Mario):
	def __init__(self, state_dim, action_dim):
		super().__init__(state_dim, action_dim)
		# optimizer 使用反向传播更新 online_q 中的参数
		self.optimizer = torch.optim.Adam(self.online_q.parameters(), lr=0.00025)

	def update_online_q(self, loss):
		'''
		输入：
		loss (torch.tensor)
			一个单值，表示 pred_q 和 target_q 的 Huber 损失
		optimizer
			optimizer 会更新我们 online_q 神经网络中的参数以降低损失
		'''
		# 更新 online_q
		self.optimizer.zero_grad()
		loss.backward()
		self.optimizer.step()

### 更新 $Q^*_{target}$

我们需要每隔一段时间将 $Q^*_{target}$ 与 $Q^*_{online}$ 同步一次，以确保我们的 $Q^*_{target}$ 是最新的。我们使用 `self.copy_every` 来控制同步频率。

In [18]:
class Mario(Mario): 
	def sync_target_q(self):
		'''
		使用在线动作价值（Q）函数更新目标动作价值（Q）函数
		'''
		self.target_q.load_state_dict(self.online_q.state_dict())

### 整合起来
辅助方法都实现之后，让我们重新回到 `Mario.learn()` 函数。

### 说明

我们已经添加了一些检查学习条件的逻辑。其余部分，请使用上面定义的辅助方法来完成 `Mario.learn()` 函数。

In [19]:
import os
import datetime

class Mario(Mario):
    def __init__(self, state_dim, action_dim):
        super().__init__(state_dim, action_dim)
        # 训练前需要收集的经验数量
        self.burnin = 1e5
        # 更新 online q 之间间隔的经验数量
        self.learn_every = 3
        # 用 online q 更新 target q 之间间隔的经验数量
        self.sync_every = 1e4
        # 保存当前智能体之间间隔的经验数量
        self.save_every = 1e5
        self.save_dir = os.path.join(
            "checkpoints",
            f"{datetime.datetime.now().strftime('%Y-%m-%dT%H-%M-%S')}"
        )
        if not os.path.exists(self.save_dir):
            os.makedirs(self.save_dir)

    def save_model(self):
        """保存当前智能体
        """
        save_path = os.path.join(self.save_dir, f"online_q_{self.step}.chkpt")
        torch.save(self.online_q.state_dict(), save_path)


    def learn(self):
        """使用一批经验更新预测动作价值（Q）函数
        """
        # 同步目标网络
        if self.step % self.sync_every == 0:
            self.sync_target_q()
        # 模型检查点
        if self.step % self.save_every == 0:
            self.save_model()
        # 如果仍处于预热阶段，则跳过
        if self.step < self.burnin:
            return
        # 如果此步不进行训练，则跳过
        if self.step % self.learn_every != 0:
            return

        # TODO：从 self.memory 中采样一批经验
        state, next_state, action, reward, done = self.sample_batch()

        # TODO：计算该批次的预测 Q 值
        pred_q = self.calculate_prediction_q(state, action)

        # TODO：计算该批次的目标 Q 值
        target_q = self.calculate_target_q(next_state, reward)

        # TODO：计算目标值与预测值之间的 huber 损失
        loss = self.calculate_loss(pred_q, target_q)
        
        # TODO：更新目标网络
        self.update_online_q(loss)
        print('正在更新')


# 开始学习！

智能体和环境包装器都实现之后，我们就可以把 Mario 放进游戏里并开始学习了！我们会用一个大的 `for` 循环来包裹学习过程，让 Mario 不断重复行动、记忆和学习。

算法的核心在循环中，让我们更仔细地看看：

### 说明

1. 在新回合开始时，我们需要调用 `state, _ = env.reset()` 来重新初始化 `state`

2. 然后我们需要几个变量来保存本回合收集到的日志信息：
  - `ep_reward`：本回合收集到的奖励
  - `ep_length`：本回合的总长度

3. 现在我们进入玩游戏的 while 循环，可以调用 `env.render()` 来显示画面

4. 现在我们要通过调用 `Mario.act(state)` 来行动。请记住，我们的动作遵循由 $Q^*_{online}$ 决定的[动作策略](https://colab.research.google.com/drive/1eN33dPVtdPViiS1njTW_-r-IYCDTFU7N#scrollTo=SZ313skqbSjQ)。

5. 通过调用 `env.step(action)` 在 env 上执行上面选出的动作。收集环境反馈：next_state、reward、terminated、truncated 和 info。

6. 通过调用 `Mario.remember(exp)` 将当前经验存入 Mario 的记忆中。

7. 通过调用 `Mario.learn()` 从 Mario 的记忆中抽取经验并更新动作策略。

8. 更新日志信息。

9. 更新 state，为下一步做准备。


In [ ]:
mario = Mario(state_dim=(4, 84, 84), action_dim=env.action_space.n)
episodes = 10000

### 通过玩游戏的方式，循环训练模型 num_episodes 次

for e in range(episodes):

    # 1. 重置 env / 重新开始游戏
    state, _ = env.reset()

    # 2. 日志记录
    ep_reward = 0.0
    ep_length = 0

    # 开始游戏！
    while True:

        # 3. 显示环境（画面）
        # env.render()
        # 4. 让智能体基于当前状态运行
        action = mario.act(state)

        # 5. 智能体执行动作
        next_state, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated

        # 6. 记住经验
        mario.remember(experience=(state, next_state, action, reward, done))

        # 7. 学习
        mario.learn()

        # 8. 日志记录
        ep_reward += reward
        ep_length += 1

        # 9. 更新状态
        state = next_state

        # 如果结束，则跳出循环
        if done or info['flag_get']:
            print(f"回合长度：{ep_length}，奖励：{ep_reward}")
            break

回合长度：266，奖励：1032.0
回合长度：40，奖励：231.0
回合长度：290，奖励：748.0
回合长度：157，奖励：637.0
回合长度：40，奖励：231.0
回合长度：311，奖励：1025.0
回合长度：632，奖励：972.0
回合长度：210，奖励：804.0
回合长度：40，奖励：238.0
回合长度：103，奖励：614.0
回合长度：171，奖励：783.0
回合长度：109，奖励：612.0
回合长度：417，奖励：1008.0
回合长度：460，奖励：994.0
回合长度：113，奖励：629.0
回合长度：557，奖励：1263.0
回合长度：555，奖励：1259.0
回合长度：273，奖励：792.0
回合长度：356，奖励：1292.0
回合长度：111，奖励：615.0
回合长度：45，奖励：218.0
回合长度：410，奖励：682.0
回合长度：344，奖励：681.0
回合长度：115，奖励：621.0
回合长度：40，奖励：231.0
回合长度：385，奖励：769.0
回合长度：168，奖励：635.0
回合长度：45，奖励：218.0
回合长度：315，奖励：1293.0
回合长度：40，奖励：231.0
回合长度：157，奖励：637.0
回合长度：216，奖励：1038.0
回合长度：205，奖励：811.0
回合长度：300，奖励：733.0
回合长度：40，奖励：231.0
回合长度：43，奖励：226.0
回合长度：300，奖励：748.0
回合长度：326，奖励：1288.0
回合长度：40，奖励：231.0
回合长度：121，奖励：594.0
回合长度：40，奖励：231.0
回合长度：399，奖励：773.0
回合长度：157，奖励：637.0
回合长度：166，奖励：615.0
回合长度：390，奖励：686.0
回合长度：117，奖励：598.0
回合长度：407，奖励：705.0
回合长度：255，奖励：1039.0
回合长度：308，奖励：733.0
回合长度：46，奖励：224.0
回合长度：191，奖励：776.0
回合长度：166，奖励：636.0
回合长度：367，奖励：1012.0
回合长度：446，奖励：1002.0
回合长度：40，奖励：231.0
回合长度：44，奖励：

# 讨论

## 离策略（Off-policy）

强化学习算法主要分为两大类：on-policy 和 off-policy。我们使用的 Q-learning 算法就是 off-policy 算法的一个例子。

这意味着 Mario 用来学习的经验，不一定需要由当前动作策略生成。Mario 可以从很久以前的记忆中学习，即使这些记忆是用已经过时的动作策略生成的。在我们的例子中，这段记忆能延伸到多“久远”，由 `Mario.max_memory` 决定。

另一方面，on-policy 算法要求 Mario 从当前动作策略生成的新鲜经验中学习。例如[策略梯度方法](https://spinningup.openai.com/en/latest/spinningup/rl_intro3.html)。


**为什么我们要从所有过去经验中采样数据点，而不是只从最近的经验中采样（例如前一个回合），尽管最近的经验是用更高准确率的新策略训练出来的？**

直觉来自这两种方法之间的权衡：

我们是希望在一个规模小但质量相对较高的数据集上训练，还是希望在一个规模巨大但质量相对较低的数据集上训练？

答案是后者，因为我们拥有的数据越多，就越能从整体、全面的视角理解系统的总体行为；在我们的例子中，就是 Mario 游戏。规模有限的数据集存在过拟合的风险，也容易忽略整个动作/状态空间中的更大图景。


请记住，强化学习的核心就是探索不同场景（state），并基于试错不断改进；这些试错来自**智能体**（action）与**环境反馈**（reward）之间的交互。


## 为什么需要两个 $Q^*$ 函数？

我们定义了两个 $Q^*$ 函数：$Q^*_{online}$ 和 $Q^*_{target}$。二者表示完全相同的东西：[最优动作价值函数（optimal value action function）](https://colab.research.google.com/drive/1eN33dPVtdPViiS1njTW_-r-IYCDTFU7N#scrollTo=snRMrCIccEx8)。我们在 [TD 估计](https://colab.research.google.com/drive/1eN33dPVtdPViiS1njTW_-r-IYCDTFU7N#scrollTo=2abP5k2kcRnn) 中使用 $Q^*_{online}$，在 [TD 目标](https://colab.research.google.com/drive/1eN33dPVtdPViiS1njTW_-r-IYCDTFU7N#scrollTo=Q072-fLecSkb) 中使用 $Q^*_{target}$。这样做是为了防止 Q-learning 更新过程中的优化发散：

$$
Q^*(s, a) \leftarrow Q^*(s, a)+\alpha (r + \gamma \max_{a'} Q^*(s', a') -Q^*(s, a))
$$

其中，第 1、第 2 和第 4 个 $Q^*(s, a)$ 使用的是 $Q^*_{online}$，而第 3 个 $Q^*(s', a')$ 使用的是 $Q^*_{target}$。


